# 🎨 Task 1: Universal Multi-Corruption Denoising Autoencoder

This notebook trains a single universal convolutional autoencoder to reconstruct clean 128x128 RGB images from clean, salt-and-pepper, gaussian blur, or rectangular occlusion inputs on Google Colab (Tesla T4 GPU).

### Pipeline Overview:
1. **Mount Drive & Set Paths**
2. **Sync Repo & Verify Authentic Image Manifests**
3. **Install Dependencies & Configure MLflow SQLite Database**
4. **Hyperparameter Selection (Fixed Optimal or Optuna Study)**
5. **Full 30-Epoch Training of 8x8x128 Bottleneck Architecture** with Cosine Annealing
6. **Comprehensive Fixed-Tier Test Evaluation** (PSNR, SSIM, L1 per severity level)
7. **ONNX Export & Numerical Parity Verification**

### Step 1: Mount Google Drive & Set Paths

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/GenAI-A1'
RAW_OXFORD_DIR = os.path.join(DRIVE_ROOT, 'raw', 'OxfordPet')
OXFORD_IMAGES_DIR = os.path.join(RAW_OXFORD_DIR, 'images_128x128')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
MLRUNS_DIR = os.path.join(DRIVE_ROOT, 'mlruns')
MANIFESTS_DIR = os.path.join(DRIVE_ROOT, 'manifests')
EXPORTS_DIR = os.path.join(DRIVE_ROOT, 'exports')
EVAL_DIR = os.path.join(DRIVE_ROOT, 'evaluation_task1')

for p in [CHECKPOINTS_DIR, MLRUNS_DIR, MANIFESTS_DIR, EXPORTS_DIR, EVAL_DIR, OXFORD_IMAGES_DIR]:
    os.makedirs(p, exist_ok=True)

print("✅ Google Drive directories configured.")

### Step 2: Clone or Update Project Repository & Validate Manifests
> **Tip:** If re-running after an update, restart the Colab runtime (`Runtime -> Restart Session`) before importing newly pulled Python modules.

In [ ]:
import sys
import json

REPO_URL = 'https://github.com/UsmanBari/genai-restoration-studio.git'
REPO_DIR = '/content/genai-restoration-studio'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Ensure authentic manifests exist and map to real extracted image files
train_m_path = os.path.join(MANIFESTS_DIR, 'oxford_train_manifest.json')
regenerate_needed = False

if not os.path.exists(train_m_path):
    regenerate_needed = True
else:
    try:
        with open(train_m_path, 'r') as f:
            data = json.load(f)
        first_fname = data[0].get('filename', '')
        first_img_path = os.path.join(OXFORD_IMAGES_DIR, first_fname)
        if not os.path.exists(first_img_path) or first_fname.startswith('pet_trainval_'):
            regenerate_needed = True
    except Exception:
        regenerate_needed = True

if regenerate_needed:
    print("⚙️ Generating authentic OxfordPet manifests directly from Drive dataset...")
    from scripts.prepare_oxford_pet import prepare_oxford_pet
    prepare_oxford_pet(
        output_dir=RAW_OXFORD_DIR,
        manifest_dir=MANIFESTS_DIR,
        target_size=(128, 128),
        split_seed=42
    )

print("✅ Working directory set to:", os.getcwd())
print("✅ Manifests ready in:", MANIFESTS_DIR)

### Step 3: Install Requirements & Configure MLflow SQLite Database

In [ ]:
!pip install -q -r requirements-colab.txt

import torch
import mlflow

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

db_path = os.path.join(MLRUNS_DIR, "mlflow.db").replace('\\', '/')
mlflow_uri = f"sqlite:///{db_path}"
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("Task1-Universal-Restoration")

print("✅ PyTorch Version:", torch.__version__)
print("✅ CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ GPU Device:", torch.cuda.get_device_name(0))
print(f"✅ MLflow SQLite Tracking URI: {mlflow_uri}")

### Step 4: Hyperparameter Configuration
We use the winning hyperparameter set discovered from Optuna tuning (`lr=0.0001225`, `batch_size=64`, `dropout_rate=0.1`, `alpha=0.95`, `base_channels=64`, `bottleneck_dim=128`).

*(Optional: Set `RUN_OPTUNA_SEARCH = True` to run an Optuna search on learning rate, batch size, base channels, dropout, and alpha with the fixed 8x8x128 bottleneck).* 

In [ ]:
RUN_OPTUNA_SEARCH = False

if RUN_OPTUNA_SEARCH:
    from training.optuna_tuner import run_optuna_study
    study = run_optuna_study(
        manifest_dir=MANIFESTS_DIR,
        images_dir=OXFORD_IMAGES_DIR,
        n_trials=12,
        trial_epochs=3,
        study_name="task1_universal_optuna",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    best_params = dict(study.best_params)
    best_params['bottleneck_dim'] = 128
else:
    # Pre-tuned winning hyperparameters
    best_params = {
        'lr': 0.0001225,
        'batch_size': 64,
        'base_channels': 64,
        'bottleneck_dim': 128,
        'dropout_rate': 0.1,
        'alpha': 0.95
    }

print("⭐ Training Hyperparameters Selected:", best_params)

### Step 5: Full 30-Epoch Training of 8x8x128 Bottleneck Architecture
Trains the updated 4-stage convolutional autoencoder ($128\times 128 \to 8\times 8\times 128$) with Cosine Annealing learning rate schedule.

In [ ]:
from models.autoencoders import UniversalAutoencoder
from data.oxford_pet import get_oxford_dataloaders
from training.trainer_universal import train_universal_autoencoder

# Dataloaders
batch_size = best_params.get('batch_size', 64)
train_loader, val_loader, test_loader = get_oxford_dataloaders(
    manifest_dir=MANIFESTS_DIR,
    images_dir=OXFORD_IMAGES_DIR,
    batch_size=batch_size,
    num_workers=2
)

# Build Model with 8x8x128 compressed bottleneck
model = UniversalAutoencoder(
    in_channels=3,
    out_channels=3,
    base_channels=best_params.get('base_channels', 64),
    bottleneck_dim=128,
    dropout_rate=best_params.get('dropout_rate', 0.1)
)

# Full 30-Epoch Training with MLflow run tracking
TOTAL_EPOCHS = 30
with mlflow.start_run(run_name="task1_universal_8x8_bottleneck_training") as run:
    mlflow.log_params(best_params)
    mlflow.log_param("total_epochs", TOTAL_EPOCHS)
    mlflow.log_param("bottleneck_spatial", "8x8")
    mlflow.log_param("bottleneck_compression_ratio", "6.0x")
    
    trained_model, train_res = train_universal_autoencoder(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=TOTAL_EPOCHS,
        lr=best_params.get('lr', 0.0001225),
        alpha=best_params.get('alpha', 0.95),
        checkpoints_dir=CHECKPOINTS_DIR,
        device="cuda" if torch.cuda.is_available() else "cpu",
        use_mlflow=True
    )

print("\n✅ Full training completed!")
print("Best Checkpoint:", train_res['best_checkpoint_path'])

### Step 6: Comprehensive Benchmark Evaluation on Test Manifest
Evaluates on all 3,669 test images across Clean, S&P, Blur, and Occlusion (low/med/high tiers).

In [ ]:
from evaluation.benchmark_universal import run_universal_benchmark

test_manifest_path = os.path.join(MANIFESTS_DIR, 'oxford_test_manifest.json')
eval_results = run_universal_benchmark(
    model=trained_model,
    manifest_path=test_manifest_path,
    images_dir=OXFORD_IMAGES_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    output_dir=EVAL_DIR,
    num_visualizations=12
)

print("\n=== Benchmark Summary Across Corruption Types ===")
for category, metrics in eval_results['grouped_by_corruption'].items():
    print(f"{category:28s} | PSNR: {metrics['psnr']:5.2f} dB | SSIM: {metrics['ssim']:5.4f} | L1: {metrics['l1']:6.5f}")

print("\n=== Detailed Severity Tier Breakdown ===")
for tier_key, metrics in eval_results['detailed_by_severity_tier'].items():
    print(f"{tier_key:32s} | PSNR: {metrics['psnr']:5.2f} dB | SSIM: {metrics['ssim']:5.4f} | L1: {metrics['l1']:6.5f}")

### Step 7: Export Model to ONNX & Verify Numerical Parity

In [ ]:
from models.onnx_export import export_to_onnx, verify_onnx_numerical_equivalence

onnx_path = os.path.join(EXPORTS_DIR, "task1_universal.onnx")
export_to_onnx(trained_model, onnx_path)

# Sample test batch for numerical equivalence
sample_batch = next(iter(test_loader))['corrupted'][:8]
parity_res = verify_onnx_numerical_equivalence(trained_model, onnx_path, sample_batch)
print("\nNumerical Verification Result:", parity_res)

### 🎉 Milestone 2 Colab Steps Complete!

**Next Action:** Download/copy the following artifacts from Google Drive (`/content/drive/MyDrive/GenAI-A1/`) to your local workspace:
1. `exports/task1_universal.onnx` $\to` `models/task1_universal.onnx`
2. `evaluation_task1/task1_benchmark_results.json` $\to` `evaluation/task1_benchmark_results.json`
3. `evaluation_task1/figures/` $\to` `evaluation/figures/`
4. Report the cell outputs back to Antigravity so the local backend and frontend can be wired up.